# arcroad · Multi-city catch-up · v0.1

**Purpose.** Bring seven more cities to the state Hyderabad reached by 26 Oct 2026, *minus everything that needs a drive*:
1. OSM extract, clipped to a collection box (dated, stored once in Drive)
2. GHSL population grid for the same box
3. Phase 1 coverage snapshot (road km, divided arterials, crossings, signals, turn restrictions)
4. Phase 2a candidate Detour Index points, using the **same `pipeline/` code and the same V10 config** as Hyderabad

**Status of every output: CANDIDATE, UNVERIFIED.** Nothing here is published, ranked or compared across cities.

**Rule 5.** This run tests no hypothesis. It is collection only. No city-vs-city comparison is made until each city has its own control and its own ground-verified route.

**Not carried over (needs a person or a drive):** ring-road sectors and rings (Phase 2c), the NH-65 baseline, the Patancheru test point, any absolute Detour Index value, anything about walking.

**How to run.** Runtime → Run all. Approve the Drive popup in cell 2. Each city saves to Drive as it finishes; if Colab disconnects, run all again and finished cities are skipped.

In [ ]:
# Cell 1 — tools and pipeline (same pinning approach as Phase 2a v0.8)
!pip -q install osmium tabulate rasterio shapely
!apt-get -qq install -y osmium-tool > /dev/null 2>&1

REPO_URL = "https://github.com/rrmlogs8896/arcroad.git"
REPO_REF = "main"          # pin a commit hash here to reproduce this run later
REPO_DIR = "/content/arcroad"

import os, subprocess, sys
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "fetch", "-q", "origin"], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", "-q", REPO_REF], check=True)
subprocess.run(["git", "-C", REPO_DIR, "pull", "-q", "--ff-only"], check=False)
sys.path.insert(0, REPO_DIR)

from pipeline import pairing, metrics, provenance
from pipeline.filters import strip_military, is_divided_proxy
SHA = pairing.commit_sha(REPO_DIR)
print("pipeline at", SHA)

In [ ]:
# Cell 2 — configuration. Drive is checked NOW, before any long download (B-13).
import datetime, json

DRIVE_BASE = "/content/drive/MyDrive/AI PROJECTS — MASTER WORKSPACE/Anthropic/Hyderabad Mobility Atlas"
MULTI_DIR  = f"{DRIVE_BASE}/multicity"       # one folder per city, one subfolder per run date
RAW_DIR    = f"{MULTI_DIR}/_raw"              # Geofabrik zone files + GHSL tiles, stored once

from google.colab import drive
drive.mount("/content/drive")
if not os.path.isdir(DRIVE_BASE):
    raise SystemExit(f"Drive folder not found: {DRIVE_BASE}\nFix DRIVE_BASE rather than creating a duplicate folder.")
os.makedirs(RAW_DIR, exist_ok=True)

# Collection boxes: (min_lon, min_lat, max_lon, max_lat). APPROXIMATE metro extents,
# sized close to Hyderabad's HMDA box. They are NOT official boundaries and must be
# replaced by a declared study area before any city's figures are used.
CITIES = {
    "bengaluru":  (77.300, 12.650, 77.950, 13.300),
    "chennai":    (79.950, 12.700, 80.350, 13.350),
    "mumbai_mmr": (72.750, 18.850, 73.250, 19.450),
    "delhi_ncr":  (76.850, 28.350, 77.550, 28.900),
    "pune":       (73.600, 18.350, 74.100, 18.750),
    "kolkata":    (88.150, 22.350, 88.600, 22.800),
    "ahmedabad":  (72.400, 22.850, 72.800, 23.200),
    # "hyderabad": pairing.HMDA_BBOX,   # uncomment to re-collect Hyderabad on the same date
}
RUN_ONLY = []            # e.g. ["pune"] to run one city; empty = all
RUN_DETOUR = True        # False = collection + coverage only (fast)

CONFIG          = pairing.V10    # identical to Hyderabad's v1.0 run
CLIP_BUFFER_DEG = 0.20           # F-3: graph clipped wider than we measure

GHSL_TILES = ("https://jeodpp.jrc.ec.europa.eu/ftp/jrc-opendata/GHSL/GHS_POP_GLOBE_R2023A/"
              "GHS_POP_E2020_GLOBE_R2023A_4326_3ss/V1-0/tiles/")
GHSL_NAME  = "GHS_POP_E2020_GLOBE_R2023A_4326_3ss_V1_0_R{r}_C{c}"

STAMP = datetime.date.today().isoformat()
todo = {k: v for k, v in CITIES.items() if not RUN_ONLY or k in RUN_ONLY}
print("run date", STAMP, "| cities:", ", ".join(todo))

In [ ]:
# Cell 3 — which Geofabrik India zones each box needs, decided from Geofabrik's own
# index geometry (not hard-coded), so a city spanning two zones (Delhi NCR) gets both.
import urllib.request
from shapely.geometry import shape, box

idx_path = f"{RAW_DIR}/geofabrik_index-v1.json"
if not os.path.exists(idx_path):
    urllib.request.urlretrieve("https://download.geofabrik.de/index-v1.json", idx_path)
index = json.load(open(idx_path))
india_zones = [f for f in index["features"]
               if "/asia/india/" in f["properties"].get("urls", {}).get("pbf", "")]
assert india_zones, "Geofabrik index has no India sub-regions — check the index file."

def zones_for(bbox):
    b = box(*bbox)
    hits = []
    for f in india_zones:
        if shape(f["geometry"]).intersects(b):
            hits.append((f["properties"]["id"], f["properties"]["urls"]["pbf"]))
    return hits

CITY_ZONES = {c: zones_for(bb) for c, bb in todo.items()}
for c, z in CITY_ZONES.items():
    print(f"{c:12s} ->", [zid for zid, _ in z])
    assert z, f"{c}: no Geofabrik zone intersects its box"

In [ ]:
# Cell 4 — helpers: download once to Drive, clip, GHSL tile calibration.
import shutil, zipfile, re, glob

def fetch_once(url, dest):
    if os.path.exists(dest) and os.path.getsize(dest) > 100_000:
        return dest
    tmp = dest + ".part"
    if subprocess.call(["wget", "-q", "--show-progress", "-O", tmp, url]) != 0:
        raise SystemExit(f"Download failed: {url}")
    os.replace(tmp, dest)
    return dest

def zone_pbf(zid, url):
    return fetch_once(url, f"{RAW_DIR}/{zid}-latest.osm.pbf")

def clip(src_paths, bbox, out):
    x0, y0, x1, y1 = bbox
    parts = []
    for i, src in enumerate(src_paths):
        p = f"/content/_part{i}.osm.pbf"
        r = subprocess.run(["osmium", "extract", "-b", f"{x0},{y0},{x1},{y1}", src, "-o", p,
                            "--overwrite", "-s", "complete_ways"], capture_output=True, text=True)
        if r.returncode != 0:
            print(r.stderr[-1500:]); raise SystemExit("Clip failed.")
        parts.append(p)
    if len(parts) == 1:
        shutil.move(parts[0], out)
    else:   # box spans zones: merge the clipped parts
        r = subprocess.run(["osmium", "merge", *parts, "-o", out, "--overwrite"],
                           capture_output=True, text=True)
        if r.returncode != 0:
            print(r.stderr[-1500:]); raise SystemExit("Merge failed.")
    return out

# GHSL tile grid: calibrated from one real tile's bounds instead of assumed.
import rasterio
from rasterio.windows import from_bounds

def ghsl_tile(r, c):
    name = GHSL_NAME.format(r=r, c=c)
    z = f"{RAW_DIR}/{name}.zip"
    try:
        fetch_once(GHSL_TILES + name + ".zip", z)
    except SystemExit:
        return None
    tifs = [n for n in zipfile.ZipFile(z).namelist() if n.endswith(".tif")]
    return f"zip://{z}!{tifs[0]}" if tifs else None

GRID = None
def calibrate():
    global GRID
    if GRID: return GRID
    # Hyderabad's tile is guessed at 10-degree tiles; its real bounds then fix the grid.
    for r, c in [(8, 26), (7, 26), (8, 25), (7, 25)]:
        t = ghsl_tile(r, c)
        if t:
            with rasterio.open(t) as ds:
                b = ds.bounds
            dx, dy = b.right - b.left, b.top - b.bottom
            GRID = dict(dx=dx, dy=dy, ox=b.left - (c - 1) * dx, oy=b.top + (r - 1) * dy)
            print("GHSL grid calibrated from", (r, c), GRID)
            return GRID
    raise SystemExit("Could not calibrate the GHSL tile grid — tell Claude.")

def ghsl_for(bbox):
    g = calibrate()
    x0, y0, x1, y1 = bbox
    rows = range(int((g["oy"] - y1) // g["dy"]) + 1, int((g["oy"] - y0) // g["dy"]) + 2)
    cols = range(int((x0 - g["ox"]) // g["dx"]) + 1, int((x1 - g["ox"]) // g["dx"]) + 2)
    return [t for r in rows for c in cols if (t := ghsl_tile(r, c))]

def ghsl_clip(bbox, out_tif):
    tiles = ghsl_for(bbox)
    assert tiles, "no GHSL tile covers this box"
    from rasterio.merge import merge
    srcs = [rasterio.open(t) for t in tiles]
    arr, transform = merge(srcs, bounds=bbox)
    meta = srcs[0].meta.copy()
    meta.update(height=arr.shape[1], width=arr.shape[2], transform=transform, compress="deflate")
    for s in srcs: s.close()
    with rasterio.open(out_tif, "w", **meta) as dst:
        dst.write(arr)
    nodata = meta.get("nodata")
    a = arr[0]
    pop = float(a[(a != nodata) & (a > 0)].sum()) if nodata is not None else float(a[a > 0].sum())
    return round(pop), len(tiles)

print("helpers loaded")

In [ ]:
# Cell 5 — Phase 1 coverage snapshot (same measures as the Hyderabad audit v0.3,
# condensed). Exploratory: it feeds no published figure.
import osmium, collections

MAIN = {"motorway","trunk","primary","secondary","tertiary","motorway_link","trunk_link",
        "primary_link","secondary_link","tertiary_link","unclassified","residential","living_street"}
ARTERIAL = {"motorway","trunk","primary","secondary"}

class Coverage(osmium.SimpleHandler):
    def __init__(self):
        super().__init__()
        self.km = collections.Counter(); self.art_km = 0.0; self.art_div_km = 0.0
        self.nodes = collections.Counter(); self.restrictions = collections.Counter()
        self.footbridges = 0; self.stripped = 0
    def node(self, n):
        hw = n.tags.get("highway")
        if hw in ("crossing", "traffic_signals"):
            self.nodes[hw] += 1
    def way(self, w):
        tags = dict(w.tags); hw = tags.get("highway")
        if hw == "footway" and tags.get("bridge") == "yes":
            self.footbridges += 1
        if hw not in MAIN:
            return
        tags, s = strip_military(tags); self.stripped += s
        try:
            m = osmium.geom.haversine_distance(w.nodes)
        except Exception:
            return
        self.km[hw] += m / 1000
        if hw in ARTERIAL:
            self.art_km += m / 1000
            if is_divided_proxy(tags): self.art_div_km += m / 1000
    def relation(self, r):
        if r.tags.get("type") == "restriction":
            v = r.tags.get("restriction") or r.tags.get("restriction:motorcar") or "unspecified"
            self.restrictions[v] += 1

def coverage(pbf):
    h = Coverage(); h.apply_file(pbf, locations=True)
    art = h.art_km or 1
    return {
        "road_km": round(sum(h.km.values()), 1),
        "arterial_km": round(h.art_km, 1),
        "arterial_divided_pct": round(100 * h.art_div_km / art, 1),
        "crossings": h.nodes["crossing"],
        "traffic_signals": h.nodes["traffic_signals"],
        "footbridges": h.footbridges,
        "turn_restrictions": sum(h.restrictions.values()),
        "u_turn_restrictions": sum(v for k, v in h.restrictions.items() if "u_turn" in k),
        "military_tagged_stripped": h.stripped,
    }
print("coverage pass loaded")

In [ ]:
# Cell 6 — the run. One city at a time; each saves to Drive before the next starts.
import time, gc, pandas as pd

summary = []
for city, bbox in todo.items():
    out = f"{MULTI_DIR}/{city}/{STAMP}"
    done_flag = f"{out}/MANIFEST.json"
    if os.path.exists(done_flag):
        print(f"[{city}] already done for {STAMP}, skipping"); summary.append(json.load(open(done_flag))); continue
    os.makedirs(out, exist_ok=True)
    t0 = time.time(); print(f"\n[{city}] box {bbox}")

    srcs = [zone_pbf(zid, url) for zid, url in CITY_ZONES[city]]
    extract_date = pairing.extract_date(srcs[0])
    b = CLIP_BUFFER_DEG
    buf = (bbox[0]-b, bbox[1]-b, bbox[2]+b, bbox[3]+b)
    clip_path = f"/content/{city}_buffered.osm.pbf"
    clip(srcs, buf, clip_path)
    shutil.copy(clip_path, f"{out}/{city}_buffered.osm.pbf")
    print(f"  OSM: {os.path.getsize(clip_path)/1e6:.1f} MB clipped, extract dated {extract_date}")

    pop, ntiles = ghsl_clip(bbox, f"{out}/{city}_ghs_pop_e2020_3ss.tif")
    print(f"  GHSL: {ntiles} tile(s), population in box ≈ {pop:,}")

    cov = coverage(clip_path)  # buffered file; box-only figures need the inner box
    print("  coverage:", cov)

    man = {"city": city, "status": "CANDIDATE — UNVERIFIED, NOT FOR PUBLICATION",
           "run_date": STAMP, "collection_box": list(bbox), "box_is_official_boundary": False,
           "clip_buffer_deg": b, "osm_zones": [z for z, _ in CITY_ZONES[city]],
           "osm_extract_date": extract_date, "commit_sha": SHA,
           "ghsl_product": "GHS_POP_E2020_GLOBE_R2023A_4326_3ss_V1_0 (CC BY 4.0)",
           "ghsl_pop_in_box": pop, "coverage_on_buffered_extract": cov,
           "attribution": "Map data © OpenStreetMap contributors (ODbL). GHSL © European Union, JRC (CC BY 4.0)."}

    if RUN_DETOUR:
        g = pairing.load_network(clip_path)
        graph = pairing.build_graph(g.ways, g.coords)
        run = pairing.run_pairing(g.ways, g.coords, config=CONFIG, zones={f"{city}_all": bbox},
                                  study_bbox=bbox, osm_extract_date=extract_date, sha=SHA, graph=graph)
        res = pd.DataFrame(run.records)
        routed = res[res.detour_index.notna() & ~res.diff_component]
        pts = f"{out}/di_points_{city}_{STAMP}_{(SHA or 'nosha')[:7]}.csv"
        if run.records:
            pairing.write_point_file(run.records, pts, run.meta)   # refuses invalid records
        else:
            print("  detour: no candidate pairs in this box — check the box before trusting it")
        h = metrics.headline_stats(routed.detour_index.tolist()) if len(routed) else {}
        man["detour_candidate"] = {"pairs_kept": len(res), "routed": len(routed),
                                   "uncensored_median": h.get("median"),
                                   "note": "ratio use only within this city; no absolute value until a drive"}
        print(f"  detour: {len(res):,} pairs, {len(routed):,} routed, uncensored median {h.get('median')}")
        del g, graph, run, res, routed; gc.collect()

    man["minutes"] = round((time.time() - t0) / 60, 1)
    json.dump(man, open(done_flag, "w"), indent=2, default=str)
    os.remove(clip_path); summary.append(man)
    print(f"  saved to {out} in {man['minutes']} min")

In [ ]:
# Cell 7 — run summary for the ledger (paste this table back to Claude).
rows = []
for m in summary:
    c = m.get("coverage_on_buffered_extract", {}); d = m.get("detour_candidate", {})
    rows.append({"city": m["city"], "extract": m["osm_extract_date"], "zones": "+".join(m["osm_zones"]),
                 "pop_in_box": m["ghsl_pop_in_box"], "road_km": c.get("road_km"),
                 "arterial_div_%": c.get("arterial_divided_pct"), "crossings": c.get("crossings"),
                 "footbridges": c.get("footbridges"), "pairs": d.get("pairs_kept"),
                 "minutes": m.get("minutes")})
tbl = pd.DataFrame(rows)
print(tbl.to_markdown(index=False))
open(f"{MULTI_DIR}/RUN_SUMMARY_{STAMP}.md", "w").write(
    f"# Multi-city catch-up {STAMP} — CANDIDATE, UNVERIFIED\n\npipeline {SHA}\n\n" + tbl.to_markdown(index=False))
print("\nNot comparable across cities: different boxes, no controls, no drives.")